# 04 - Gold Layer

**Objetivo:** Criar a camada Gold da arquitetura Medallion, pronta para consumo final no Power BI.

Nesta etapa os dados da camada Silver (já modelados em Star Schema com colunas de ordenação) são unidos em uma tabela denormalizada otimizada, facilitando a importação e criação de visuais no Power BI.

### O que é feito nesta camada:
- Leitura das tabelas da camada Silver
- Junção final (denormalizada) para criar uma única tabela `gold_credito` com todas as informações
- Salvamento de todas as tabelas (dimensões + fato) em formato Parquet na pasta `data/gold/`

**Importante:** Nesta camada **não** são criadas novas métricas nem colunas de ordenação (elas já estão nas dimensões da Silver).  
O foco é performance e simplicidade para o Power BI.

**Próxima etapa:** Importar a pasta `data/gold/` no Power BI e criar os relacionamentos + dashboard.

In [17]:
import pandas as pd
from pathlib import Path

In [18]:
# ============================
# 04 - Gold Layer
# ============================


# ==================== CONFIGURAÇÃO ====================
silver_dir = Path('../data/silver')
gold_dir = Path('../data/gold')
gold_dir.mkdir(parents=True, exist_ok=True)

# ==================== 1. CARREGAR SILVER ====================
dim_tempo = pd.read_parquet(silver_dir / 'dim_tempo.parquet')
dim_produto = pd.read_parquet(silver_dir / 'dim_produto.parquet')
dim_consumidor = pd.read_parquet(silver_dir / 'dim_consumidor.parquet')
fact_credito = pd.read_parquet(silver_dir / 'fact_credito.parquet')

print(f"Silver carregada com sucesso!")

# ==================== 2. CRIAR TABELA GOLD (AGREGADA) ====================


Silver carregada com sucesso!


In [19]:


descricao = {'emprestimo_pessoal_fgts':'Emp. Pes. Fgts',
              'conta_digital':'Conta Digital',
              'emprestimo_pessoal':'Emp. Pesssoal',
              'emprestimo_garantia_veiculo':'Emp. Gar. Vei.',
              'cartao_credito':'Cartão Crédito',
              'emprestimo_consignado':'Emp. Consignado',
              'emprestimo_garantia_imovel':'Emp. Gar. Imó.',}

dim_produto['descricao'] = dim_produto['tipo_credito'].map(descricao)

In [20]:
# ==================== 3. CRIACAO DE FEATURES (SEM DIM_TEMPO - evita dados espaços) ====================

# Agrega removendo o grão temporal (equivalente a SUM ... PARTITION BY sk_produto, sk_consumidor)
fact_credito_agg = (
    fact_credito
    .groupby(['sk_produto', 'sk_consumidor'], as_index=False)
    .agg({
        'total_usuarios_simulando': 'sum',
        'usuarios_elegiveis': 'sum',
        'possui_oferta': 'sum',
        'conversao_efetiva': 'sum',
        'receita_gerada': 'sum'
    })
)

# Recalcula features no novo grão
fact_credito_agg['taxa_efetividade'] = (
    fact_credito_agg['usuarios_elegiveis']
    .div(fact_credito_agg['total_usuarios_simulando'].replace(0, pd.NA))
    .fillna(0)
)

fact_credito_agg.drop(columns=['total_usuarios_simulando'],inplace=True)

fact_credito_agg['taxa_ofertas'] = (
    fact_credito_agg['possui_oferta']
    .div(fact_credito_agg['usuarios_elegiveis'].replace(0, pd.NA))
    .fillna(0)
)

fact_credito_agg.drop(columns=['usuarios_elegiveis'],inplace=True)

fact_credito_agg['taxa_conversao'] = (
    fact_credito_agg['conversao_efetiva']
    .div(fact_credito_agg['possui_oferta'].replace(0, pd.NA))
    .fillna(0)
)

fact_credito_agg.drop(columns=['possui_oferta'],inplace=True)

fact_credito_agg['ticket_medio'] = (
    fact_credito_agg['receita_gerada']
    .div(fact_credito_agg['conversao_efetiva'].replace(0, pd.NA))
    .fillna(0)
)

fact_credito_agg.drop(columns=['conversao_efetiva'],inplace=True)
fact_credito_agg.drop(columns=['receita_gerada'],inplace=True)

fact_credito = fact_credito.merge(fact_credito_agg, on=['sk_produto', 'sk_consumidor'], how='left')


C:\Users\eduar\AppData\Local\Temp\ipykernel_22576\2027985208.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0)
C:\Users\eduar\AppData\Local\Temp\ipykernel_22576\2027985208.py:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0)


In [22]:
# ==================== 3. SALVAR NA GOLD ====================
fact_credito.to_parquet(gold_dir / 'gold_credito.parquet', index=False)

# Salvar também as dimensões (facilita import único no Power BI)
dim_tempo.to_parquet(gold_dir / 'dim_tempo.parquet', index=False)
dim_produto.to_parquet(gold_dir / 'dim_produto.parquet', index=False)
dim_consumidor.to_parquet(gold_dir / 'dim_consumidor.parquet', index=False)

print("Camada Gold criada com sucesso!")
print(f"gold_credito: {len(fact_credito):,} linhas")
print("Dimensões salvas na Gold")

Camada Gold criada com sucesso!
gold_credito: 105,651 linhas
Dimensões salvas na Gold
